# 02 - IU X-ray Preparation

**Why IU X-ray:** ~7,470 images with paired Findings + Impression text. NIH has labels but no reports — and our project requires generating reports, so we **must** evaluate against ground-truth reports somewhere. IU X-ray is the smallest, fastest, fully public dataset with paired reports.

**This notebook:**
1. Downloads the Indiana University CXR dataset from Kaggle.
2. Cleans the reports (removes XML artefacts, anonymisation tokens like `XXXX`).
3. Pseudo-labels each report against the 14 NIH classes (regex labeller).
4. Builds a held-out test split for the RAG eval.
5. Saves a CSV that the RAG/eval notebooks consume.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Upload kaggle.json once. Format: {"username": "...", "key": "..."}
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\n  "username": "ramzieee",\n  "key": "KGAT_5155b25ddaf556fe30cfc4b4726e5e17"\n}\n'}

In [3]:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d raddar/chest-xrays-indiana-university -p /content --quiet
!unzip -q -o /content/chest-xrays-indiana-university.zip -d /content/iu_data
!ls /content/iu_data

Dataset URL: https://www.kaggle.com/datasets/raddar/chest-xrays-indiana-university
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
images	indiana_projections.csv  indiana_reports.csv


In [14]:
import os, re, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = '/content/iu_data'
IMG_DIR  = os.path.join(DATA_DIR, 'images', 'images_normalized')
OUT_DIR  = '/content/drive/MyDrive/Data/iu_processed'
os.makedirs(OUT_DIR, exist_ok=True)

reports = pd.read_csv(os.path.join(DATA_DIR, 'indiana_reports.csv'))
projections = pd.read_csv(os.path.join(DATA_DIR, 'indiana_projections.csv'))
print('reports:', reports.shape, '|', list(reports.columns))
print('projections:', projections.shape, '|', list(projections.columns))

reports: (3851, 8) | ['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']
projections: (7466, 3) | ['uid', 'filename', 'projection']


## 1. Clean reports

IU reports contain things like `XXXX` (anonymised tokens) and inconsistent whitespace. Strip those and the obvious boilerplate.

In [15]:
def clean_text(t):
    if not isinstance(t, str):
        return ''
    t = re.sub(r'X{2,}', '', t)            # anonymisation tokens
    t = re.sub(r'\s+', ' ', t).strip()      # collapse whitespace
    return t

for col in ['indication', 'comparison', 'findings', 'impression']:
    if col in reports.columns:
        reports[col] = reports[col].apply(clean_text)

# Keep only rows with non-trivial findings AND impression
reports = reports[(reports['findings'].str.len() > 10) & (reports['impression'].str.len() > 5)].copy()
print('Reports after filtering:', len(reports))

Reports after filtering: 3330


## 2. Pseudo-label reports against the 14 NIH classes

Regex-based labeller. Lets us compute label-F1 against IU reports and gives the retriever a label-conditioned signal even when the classifier hasn't seen IU images.

In [16]:
CLASS_NAMES = ['Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion','Emphysema','Fibrosis','Hernia','Infiltration','Mass','Nodule','Pleural_Thickening','Pneumonia','Pneumothorax']

POS_PATTERNS = {
    'Atelectasis':         [r'\batelectas',  r'\bcollapse\b'],
    'Cardiomegaly':        [r'cardiomegaly', r'enlarged heart', r'cardiac enlargement', r'cardiothoracic ratio'],
    'Consolidation':       [r'consolidat'],
    'Edema':               [r'\bedema\b', r'pulmonary edema', r'kerley'],
    'Effusion':            [r'effusion', r'pleural fluid'],
    'Emphysema':           [r'emphysema', r'hyperinflat'],
    'Fibrosis':            [r'fibrosis', r'fibrotic', r'honeycomb'],
    'Hernia':              [r'hernia'],
    'Infiltration':        [r'infiltrat'],
    'Mass':                [r'\bmass\b'],
    'Nodule':              [r'nodule'],
    'Pleural_Thickening':  [r'pleural thickening'],
    'Pneumonia':           [r'pneumonia'],
    'Pneumothorax':        [r'pneumothorax'],
}

NEG_RE = re.compile(r'\b(no|without|absence of|negative for|free of|denies|rule out|r/o)\b')

def is_negated(text, span_start):
    """Cheap negation detection: any negation word within 60 chars left of the term."""
    window = text[max(0, span_start-60):span_start]
    return bool(NEG_RE.search(window))

def encode_labels(text):
    text = text.lower()
    out = [0] * len(CLASS_NAMES)
    for i, c in enumerate(CLASS_NAMES):
        for pat in POS_PATTERNS[c]:
            for m in re.finditer(pat, text):
                if not is_negated(text, m.start()):
                    out[i] = 1
                    break
            if out[i]:
                break
    return out

reports['report_text'] = (reports['findings'].fillna('') + ' ' + reports['impression'].fillna('')).str.strip()
reports['encoded_labels'] = reports['report_text'].apply(encode_labels)
label_mat = np.array(reports['encoded_labels'].tolist())
print('Per-class positive count in IU reports:')
for c, n in zip(CLASS_NAMES, label_mat.sum(0).tolist()):
    print(f'  {c:22s} {n}')
print('Total reports labelled:', len(reports))

Per-class positive count in IU reports:
  Atelectasis            296
  Cardiomegaly           194
  Consolidation          51
  Edema                  63
  Effusion               347
  Emphysema              139
  Fibrosis               22
  Hernia                 38
  Infiltration           80
  Mass                   25
  Nodule                 84
  Pleural_Thickening     24
  Pneumonia              70
  Pneumothorax           158
Total reports labelled: 3330


## 3. Merge reports with image projections (front-only for cleanliness)

In [17]:
DRIVE_IMG_DIR = '/content/drive/MyDrive/Data/iu_images'
# Copy images to Drive ONCE so paths survive across Colab sessions
import os
if not os.path.exists(DRIVE_IMG_DIR) or len(os.listdir(DRIVE_IMG_DIR)) == 0:
    !mkdir -p {DRIVE_IMG_DIR}
    !cp -r {IMG_DIR}/. {DRIVE_IMG_DIR}/

df = reports.merge(projections, on='uid', how='inner')
df = df[df['projection'].str.lower().eq('frontal')].copy()  # PA/AP frontal only
# Store the persistent DRIVE path in the CSV so notebooks 03 and 04 can find images
# after a Colab restart (the /content/iu_data/ path is wiped between sessions).
df['image_path'] = df['filename'].apply(lambda x: os.path.join(DRIVE_IMG_DIR, x))
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print('Final paired (image, report) rows:', len(df))
print(df[['uid', 'filename', 'findings']].head(2))

Final paired (image, report) rows: 3300
   uid                filename  \
0    1  1_IM-0001-4001.dcm.png   
1    2  2_IM-0652-1001.dcm.png   

                                            findings  
0  The cardiac silhouette and mediastinum size ar...  
1  Borderline cardiomegaly. Midline sternotomy . ...  


## 4. Patient-level split (no leakage)

Multiple images can come from the same study/patient → split by `uid` so the same patient never appears in both train and test.

In [18]:
uids = df['uid'].unique()
rng = np.random.default_rng(42)
rng.shuffle(uids)
n = len(uids); n_te = int(0.15 * n); n_va = int(0.10 * n)
test_uids  = set(uids[:n_te])
val_uids   = set(uids[n_te:n_te+n_va])
train_uids = set(uids[n_te+n_va:])

def part(u):
    if u in test_uids:  return 'test'
    if u in val_uids:   return 'val'
    return 'train'
df['split'] = df['uid'].apply(part)
print(df['split'].value_counts())

df.to_csv(os.path.join(OUT_DIR, 'iu_pairs.csv'), index=False)
df[df['split']=='train'].to_csv(os.path.join(OUT_DIR, 'iu_train.csv'), index=False)
df[df['split']=='val'  ].to_csv(os.path.join(OUT_DIR, 'iu_val.csv'),   index=False)
df[df['split']=='test' ].to_csv(os.path.join(OUT_DIR, 'iu_test.csv'),  index=False)

# (Image copy already done above so paths in CSV are persistent.)
print('Saved IU splits to', OUT_DIR)
print('Test set size for RAG eval:', (df['split']=='test').sum())

split
train    2474
test      501
val       325
Name: count, dtype: int64
Saved IU splits to /content/drive/MyDrive/Data/iu_processed
Test set size for RAG eval: 501


## 5. Sanity check — show one example

In [19]:
row = df[df['split']=='test'].iloc[0]
print('Image:', row['image_path'])
print('\nFindings:\n', row['findings'])
print('\nImpression:\n', row['impression'])
print('\nLabels:', dict(zip(CLASS_NAMES, row['encoded_labels'])))

Image: /content/drive/MyDrive/Data/iu_images/2_IM-0652-1001.dcm.png

Findings:
 Borderline cardiomegaly. Midline sternotomy . Enlarged pulmonary arteries. Clear lungs. Inferior .

Impression:
 No acute pulmonary findings.

Labels: {'Atelectasis': 0, 'Cardiomegaly': 1, 'Consolidation': 0, 'Edema': 0, 'Effusion': 0, 'Emphysema': 0, 'Fibrosis': 0, 'Hernia': 0, 'Infiltration': 0, 'Mass': 0, 'Nodule': 0, 'Pleural_Thickening': 0, 'Pneumonia': 0, 'Pneumothorax': 0}


In [22]:
row = df[df['split']=='test'].iloc[4]
print('Image:', row['image_path'])
print('\nFindings:\n', row['findings'])
print('\nImpression:\n', row['impression'])
print('\nLabels:', dict(zip(CLASS_NAMES, row['encoded_labels'])))

Image: /content/drive/MyDrive/Data/iu_images/34_IM-1644-1001.dcm.png

Findings:
 The heart is normal in size and contour. The lungs are clear, without evidence of infiltrate. There is no pneumothorax or effusion.

Impression:
 No acute cardiopulmonary disease.

Labels: {'Atelectasis': 0, 'Cardiomegaly': 0, 'Consolidation': 0, 'Edema': 0, 'Effusion': 0, 'Emphysema': 0, 'Fibrosis': 0, 'Hernia': 0, 'Infiltration': 0, 'Mass': 0, 'Nodule': 0, 'Pleural_Thickening': 0, 'Pneumonia': 0, 'Pneumothorax': 0}


In [23]:
!ls '/content/drive/MyDrive/Data/iu_processed'

iu_pairs.csv  iu_test.csv  iu_train.csv  iu_val.csv


In [25]:
!cp -r '/content/drive/MyDrive/Data/iu_processed' '/content/drive/MyDrive/Data/iu_processed/'

cp: cannot copy a directory, '/content/drive/MyDrive/Data/iu_processed', into itself, '/content/drive/MyDrive/Data/iu_processed/iu_processed'


In [28]:
!mkdir '/content/drive/MyDrive/Data/iu_content'
!cp -r '/content/drive/MyDrive/Data/iu_processed' '/content/drive/MyDrive/Data/iu_content'

mkdir: cannot create directory ‘/content/drive/MyDrive/Data/iu_content’: File exists
